# Incident Triage Playbook\n\nSimulate a first-pass investigation: pull search data, check control-plane context, chart a signal, then draft AI-assisted findings.

## Investigation flow\n\n1. Pull suspicious recent events (`%%cribl_search`)\n2. Inspect search job/API status (`%%cribl_api`)\n3. Build a quick chart\n4. Use AI prompt template to draft incident notes\n\n**Run All** uses **`lang=kql`** so this playbook works on every hosted tenant. For natural-language → KQL (requires leader AI), see `Cribl_Search_Examples.ipynb`.

In [ ]:
%%cribl_search var=incident_df lang=kql dataset=cribl_search_sample earliest=-1h latest=now limit=50\ndataset=cribl_search_sample\n| sort by _time desc\n| limit 50

In [ ]:
%%cribl_api GET /m/default_search/search/jobs var=job_inventory response=json\nheaders:\n  Accept: application/json

In [ ]:
print('incident rows:', len(incident_df))\nprint('job inventory type:', type(job_inventory))\nincident_df.head(10)

In [ ]:
import matplotlib.pyplot as plt\n\nif len(incident_df) > 0:\n    metric_candidates = [c for c in ['bytes', 'latency_ms', 'duration_ms'] if c in incident_df.columns]\n    if metric_candidates:\n        metric = metric_candidates[0]\n        series = incident_df[metric].astype(float)\n        series.head(200).plot(kind='line', title=f'Incident signal: {metric}')\n        plt.tight_layout()\n        plt.show()\n    else:\n        print('No standard incident metric columns found; available:', list(incident_df.columns)[:20])\nelse:\n    print('No rows returned')

In [ ]:
# ### Prompt:\n# Create a short incident summary using notebook objects below.\n#\n# Search context: {{ incident_df | describe }}\n# API context: {{ job_inventory | describe }}\n#\n# Requirements:\n# - list top 3 notable findings\n# - include one potential false-positive caveat\n# - suggest one concrete next query to run

## Debrief\n\nThis playbook is intentionally compact. For **threat hunting** with a static IP lookup, KQL **join**, and **`timestats`** timeline chart on `cribl_search_sample`, open `Threat_Hunting_Playbook.ipynb`. For deeper API operations, open `Cribl_API_Examples.ipynb`; for richer prompt iteration, open `AI_Magic.ipynb`.